In [0]:
-- ============================================================
-- Szenario B: Neuer Kunde (Priya) legt eine Bestellung auf --
-- Nachweis, dass die Pipeline auch komplett neue Kunden korrekt
-- durch alle drei Layer schleust (nicht nur SCD2-Änderungen)
-- ============================================================

-- Neuer Kunde in der Quelle
INSERT INTO brewflow_oltp.customers VALUES
(11, 'Priya', 'Sharma', 'priya.s@example.com', 'Mumbai', 'Bronze', current_date());

-- Neue Bestellung + Position dafür
INSERT INTO brewflow_oltp.orders VALUES
(201, 11, 1, current_date(), 'mobile-app', 3.80);

INSERT INTO brewflow_oltp.order_items VALUES
(2001, 201, 2, 1, 3.80);  -- 1x Cappuccino

-- Validieren: OLTP jetzt 11 Kunden, 23 Orders, 33 Order Items
SELECT 'customers' AS tbl, COUNT(*) AS rows FROM brewflow_oltp.customers
UNION ALL SELECT 'orders',      COUNT(*) FROM brewflow_oltp.orders
UNION ALL SELECT 'order_items', COUNT(*) FROM brewflow_oltp.order_items;

-- ============================================================
-- JETZT: Pipeline erneut ausführen! (03 -> 04 -> 05 bzw. Lakeflow Job)
-- ============================================================

-- Nachher-Zustand prüfen
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT customer_id) AS distinct_customers
FROM brewflow_warehouse.dim_customer;
-- Erwartet: total_rows = 12, distinct_customers = 11 (Sofia weiterhin mit 2 Versionen)

SELECT COUNT(*) AS fact_rows FROM brewflow_warehouse.fact_order_items;
-- Erwartet: 33 (32 Baseline + 1 neue Position)

-- Priyas neue Bestellung im Fact nachverfolgen
SELECT f.order_item_id, f.order_id, dc.first_name, dc.loyalty_tier, dp.name AS product, f.quantity, f.unit_price
FROM brewflow_warehouse.fact_order_items f
JOIN brewflow_warehouse.dim_customer dc ON f.customer_sk = dc.customer_sk
JOIN brewflow_warehouse.dim_product  dp ON f.product_sk  = dp.product_sk
WHERE dc.customer_id = 11;
-- Erwartet: 1 Zeile, Priya, Bronze, Cappuccino, Menge 1